# Web Scraping Yahoo Finance

In this notebook, we will demonstrate how to scrape financial data from Yahoo Finance using Python. This includes extracting stock prices, historical data, and other relevant financial metrics.

In [5]:
# Import Required Libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import re
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

In [6]:
# Define a function to scrape stock data from Yahoo Finance
def scrape_yahoo_finance(stock_symbol):
    """Scrape historical stock data from Yahoo Finance"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    url = f"https://finance.yahoo.com/quote/{stock_symbol}/history?p={stock_symbol}"
    
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()  # Raises an HTTPError for bad responses
        
        soup = BeautifulSoup(response.content, 'html.parser')
        table = soup.find('table', {'data-test': 'historical-prices'})
        
        if table:
            rows = table.find_all('tr')
            data = []
            
            for row in rows[1:]:  # Skip header row
                cols = row.find_all('td')
                if len(cols) >= 6:  # Ensure the row has minimum required columns
                    row_data = [col.text.strip() for col in cols[:7]]  # Take first 7 columns
                    # Skip dividend rows (they have different structure)
                    if 'Dividend' not in row_data[0]:
                        data.append(row_data)
            
            if data:
                df = pd.DataFrame(data, columns=['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume'])
                # Clean and convert data types
                df = clean_stock_data(df)
                return df
            else:
                print(f"No historical data found for {stock_symbol}")
                return None
        else:
            print(f"Could not find historical prices table for {stock_symbol}")
            return None
            
    except requests.RequestException as e:
        print(f"Error fetching data for {stock_symbol}: {e}")
        return None
    except Exception as e:
        print(f"Unexpected error: {e}")
        return None

In [7]:
def clean_stock_data(df):
    """Clean and convert stock data to appropriate types"""
    # Convert Date column
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Clean numeric columns (remove commas and convert to float)
    numeric_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    
    for col in numeric_cols:
        if col in df.columns:
            # Remove commas and handle '-' values
            df[col] = df[col].str.replace(',', '').str.replace('-', '0')
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Sort by date (newest first to oldest)
    df = df.sort_values('Date', ascending=False).reset_index(drop=True)
    
    return df

In [8]:
# Example usage of the scraping function
stock_symbol = 'AAPL'  # Apple Inc.
print(f"Fetching historical data for {stock_symbol}...")

data = scrape_yahoo_finance(stock_symbol)

if data is not None:
    print(f"\nSuccessfully retrieved {len(data)} records for {stock_symbol}")
    print("\nFirst 5 records:")
    print(data.head())
    
    print("\nData types:")
    print(data.dtypes)
    
    print("\nBasic statistics:")
    print(data[['Open', 'High', 'Low', 'Close', 'Volume']].describe())
else:
    print("Failed to retrieve data.")

Fetching historical data for AAPL...


Could not find historical prices table for AAPL
Failed to retrieve data.


In [9]:
def get_current_stock_info(stock_symbol):
    """Get current stock price and key metrics from Yahoo Finance"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    url = f"https://finance.yahoo.com/quote/{stock_symbol}"
    
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Extract current price
        price_element = soup.find('fin-streamer', {'data-field': 'regularMarketPrice'})
        current_price = price_element.text if price_element else 'N/A'
        
        # Extract change
        change_element = soup.find('fin-streamer', {'data-field': 'regularMarketChange'})
        change = change_element.text if change_element else 'N/A'
        
        # Extract change percentage
        change_pct_element = soup.find('fin-streamer', {'data-field': 'regularMarketChangePercent'})
        change_pct = change_pct_element.text if change_pct_element else 'N/A'
        
        # Extract company name
        name_element = soup.find('h1', {'data-reactroot': ''})
        company_name = name_element.text.split('(')[0].strip() if name_element else stock_symbol
        
        return {
            'Symbol': stock_symbol,
            'Company': company_name,
            'Current Price': current_price,
            'Change': change,
            'Change %': change_pct,
            'Last Updated': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        
    except Exception as e:
        print(f"Error getting current info for {stock_symbol}: {e}")
        return None

In [10]:
# Get current stock information
print("\n" + "="*50)
print("CURRENT STOCK INFORMATION")
print("="*50)

current_info = get_current_stock_info('AAPL')
if current_info:
    for key, value in current_info.items():
        print(f"{key}: {value}")
else:
    print("Failed to retrieve current stock information")


CURRENT STOCK INFORMATION
Symbol: AAPL
Company: AAPL
Current Price: 6,000.36
Change: +61.06
Change %: (+1.03%)
Last Updated: 2025-06-08 12:33:10
Symbol: AAPL
Company: AAPL
Current Price: 6,000.36
Change: +61.06
Change %: (+1.03%)
Last Updated: 2025-06-08 12:33:10


In [11]:
def scrape_multiple_stocks(stock_symbols, delay=1):
    """Scrape historical data for multiple stocks"""
    all_data = {}
    
    for symbol in stock_symbols:
        print(f"Fetching data for {symbol}...")
        data = scrape_yahoo_finance(symbol)
        
        if data is not None:
            all_data[symbol] = data
            print(f"✓ Successfully retrieved {len(data)} records for {symbol}")
        else:
            print(f"✗ Failed to retrieve data for {symbol}")
        
        # Add delay to be respectful to the server
        time.sleep(delay)
    
    return all_data

# Example: Get data for multiple tech stocks
tech_stocks = ['AAPL', 'GOOGL', 'MSFT', 'TSLA']
print("\n" + "="*50)
print("SCRAPING MULTIPLE STOCKS")
print("="*50)

stock_data = scrape_multiple_stocks(tech_stocks)

print(f"\nSuccessfully retrieved data for {len(stock_data)} out of {len(tech_stocks)} stocks")
for symbol, data in stock_data.items():
    print(f"{symbol}: {len(data)} records (from {data['Date'].min().date()} to {data['Date'].max().date()})")


SCRAPING MULTIPLE STOCKS
Fetching data for AAPL...
Could not find historical prices table for AAPL
✗ Failed to retrieve data for AAPL
Could not find historical prices table for AAPL
✗ Failed to retrieve data for AAPL
Fetching data for GOOGL...
Fetching data for GOOGL...
Could not find historical prices table for GOOGL
✗ Failed to retrieve data for GOOGL
Could not find historical prices table for GOOGL
✗ Failed to retrieve data for GOOGL
Fetching data for MSFT...
Fetching data for MSFT...
Could not find historical prices table for MSFT
✗ Failed to retrieve data for MSFT
Could not find historical prices table for MSFT
✗ Failed to retrieve data for MSFT
Fetching data for TSLA...
Fetching data for TSLA...
Could not find historical prices table for TSLA
✗ Failed to retrieve data for TSLA
Could not find historical prices table for TSLA
✗ Failed to retrieve data for TSLA

Successfully retrieved data for 0 out of 4 stocks

Successfully retrieved data for 0 out of 4 stocks


In [12]:
# Simple data analysis and visualization
import matplotlib.pyplot as plt

if 'AAPL' in stock_data:
    apple_data = stock_data['AAPL'].copy()
    
    # Plot closing price over time
    plt.figure(figsize=(12, 6))
    plt.plot(apple_data['Date'], apple_data['Close'], linewidth=2, color='blue')
    plt.title('Apple (AAPL) Stock Price - Closing Price Over Time', fontsize=16, fontweight='bold')
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Closing Price ($)', fontsize=12)
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Calculate and display some basic metrics
    print("\n" + "="*50)
    print("APPLE STOCK ANALYSIS")
    print("="*50)
    
    current_price = apple_data['Close'].iloc[0]
    highest_price = apple_data['Close'].max()
    lowest_price = apple_data['Close'].min()
    avg_volume = apple_data['Volume'].mean()
    
    print(f"Current Closing Price: ${current_price:.2f}")
    print(f"52-Week High: ${highest_price:.2f}")
    print(f"52-Week Low: ${lowest_price:.2f}")
    print(f"Average Volume: {avg_volume:,.0f}")
    
    # Calculate moving averages
    apple_data['MA_7'] = apple_data['Close'].rolling(window=7).mean()
    apple_data['MA_30'] = apple_data['Close'].rolling(window=30).mean()
    
    print(f"7-Day Moving Average: ${apple_data['MA_7'].iloc[0]:.2f}")
    print(f"30-Day Moving Average: ${apple_data['MA_30'].iloc[0]:.2f}")
else:
    print("Apple data not available for analysis")

Apple data not available for analysis
